In [ ]:
# Storing Spotify Client ID for API authentication
CLIENT_ID ='xxxxxxxxxxxxxxxxxxxxxxxx'

# Storing Spotify Client Secret for API authentication
SECRET_ID ='xxxxxxxxxxxxxxxxxxxxxxxx'

### Explanation:
* CLIENT_ID and SECRET_ID are credentials you get from the Spotify Developer Dashboard when you create a new app.

* These are used to authenticate your app with Spotify's API using the Client Credentials Flow.

* Once authenticated, you'll receive an access token that allows you to make requests to Spotify’s API (like searching playlists, getting track data, etc.).

* Without these credentials, your requests to the API will be unauthorized.

In [4]:
# Import necessary libraries
import requests
import pandas as pd

In [5]:
# Define a function to get the Spotify API access token
def get_token(): 
     # Store your Spotify Client ID and Spotify Secret ID
    client_id = CLIENT_ID
    client_secret= SECRET_ID
    
    # Send POST request to get access token using client credentials
    auth_response = requests.post(
        "https://accounts.spotify.com/api/token",
        
        # Set grant_type to client_credentials for this kind of authentication
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret)
    )
    
    # Extract the access token from the response JSON
    access_token = auth_response.json().get("access_token")
    return access_token

# Print the token (for testing purposes only)
print(get_token())

BQAjXcdIgxFLwHBge4TV_qXvIDwQquIaFWdnBSnAAanr4KMbnQJ5OKjEXye7yvVoXWa6IM30QKYQQ4cr2mWRxvdoPR5mYwv0mAfeAWnMCdwtHSEFOi2TenFVV29u2reOMFW08HyhVzU


In [6]:
# Define a function to search playlists from Spotify API
def search_playlists(query, token, limit=10, offset=0):

    # Spotify Search endpoint
    url = "https://api.spotify.com/v1/search"

    # Add authorization token to the request headers
    headers = {"Authorization": f"Bearer {token}"}

    # Set the search parameters: keyword, type, limit, and market
    params = {
        "q": query,          # Search keyword
        "type": "playlist",  # We're only searching for playlists
        "limit": limit,      # Max number of playlists to return
        "market": "US",      # Target market
        "offset": offset     # For pagination (skip items)
    }

    # Send the GET request to Spotify API with the headers and params
    res = requests.get(url, headers=headers, params=params)

    # If the request failed, print error and return empty list
    if res.status_code != 200:
        print("Failed to fetch playlists:", res.status_code)
        print(res.text)
        return []

        # Parse the JSON response and extract playlist items
    playlists = res.json().get("playlists", {}).get("items", [])

    # Initialize list to hold final playlist results
    results = []


        # Loop through each playlist item
    for p in playlists:
        if not p:
            continue  # Skip if playlist is None

        # Extract needed fields: id, name, and Spotify URL
        playlist_id = p.get("id")
        name = p.get("name")
        url = p.get("external_urls", {}).get("spotify")

        # Add playlist info to results
        results.append({"id": playlist_id, "name": name, "url": url})

    # Return list of formatted playlist data
    return results



In [8]:
# Get a fresh access token using the get_token() function
token = get_token()

# Search for playlists that match the keyword "hip hop usa", limit results to 5
results = search_playlists("hip hop usa", token, limit=20)

# Loop through each playlist in the search results
for r in results:
    # Print the playlist name and its Spotify URL
    print(f"{r['name']} - {r['url']}")


USA Rap - 2025 🔥 - https://open.spotify.com/playlist/2JohRIjpua2FHY0skj7CSt
4th of July Hip Hop and R&B  Indepence Day, Fireworks, Food, and Fun - https://open.spotify.com/playlist/2XXie4IVyrwQwxlK0aPuKS
hip hop USA 2021/25 - https://open.spotify.com/playlist/0PaSxJgDBs2uLLwDL9Igjb
Dance Mix USA Vol 1 - https://open.spotify.com/playlist/0fHECQdMuE5dEQ1cRqUUFj
english trap/rap - https://open.spotify.com/playlist/7HADjFUYUmuOzuW06La57t
USA  Rap & Hip-Hop (2000-2010) - https://open.spotify.com/playlist/60fG165rIvVpopOgYOr8OX
POP SMOKE HARDEST SONGS  - https://open.spotify.com/playlist/6DiV3nGCuRQbF0XkSWOwFR
USA rap 2010-2022 - https://open.spotify.com/playlist/2rx9hqcpkaex44OIqjBxUf
4th of July Hip Hop - https://open.spotify.com/playlist/0oBsVbv3rsZhWpS46ok4JJ
Best of HIP HOP - USA - https://open.spotify.com/playlist/6VPT01v7vDDQdN3HyuwvRk
SONGS ABOUT USA - https://open.spotify.com/playlist/4SN0YNZ6sF1nf26mNXv2Bi
Dance Mix USA Vol 4 - https://open.spotify.com/playlist/2YTEQtte1DwiWhKp9KGb

In [16]:
# Define a function to fetch detailed information about a playlist using its ID
def get_playlist_detail(playlist_id, token):
    # Construct the Spotify API endpoint URL for the playlist
    url = f"https://api.spotify.com/v1/playlists/{playlist_id}"

    # Prepare the headers with the access token for authorization
    headers = {"Authorization": f"Bearer {token}"}
    
    # Send a GET request to the API to fetch playlist details
    res = requests.get(url, headers=headers)

    # If the request failed (e.g., invalid token or ID), print an error and return None
    if res.status_code != 200:
        print(f"Gagal ambil detail untuk {playlist_id}")
        return None

    # Convert the response to JSON
    data = res.json()

    # Return selected details: description and number of followers (saves)
    return {
        "description": data.get("description", ""),
        "followers": data.get("followers", {}).get("total", 0)
    }


In [18]:
# Save to CSV
query = "hip hop usa"
playlist_data = search_playlists(query, token, limit=10)

final_results = []

for p in playlist_data:
    detail = get_playlist_detail(p["id"], token)
    if not detail:
        continue
    
    final_results.append({
        "title": p["name"],
        "url": p["url"],
        "description": detail["description"],
        "saves": detail["followers"]
    })
    
df = pd.DataFrame(final_results)
print(df)

                                               title  \
0                                   USA Rap - 2025 🔥   
1  4th of July Hip Hop and R&B  Indepence Day, Fi...   
2                                hip hop USA 2021/25   
3                                Dance Mix USA Vol 1   
4                                   english trap/rap   
5                     USA  Rap & Hip-Hop (2000-2010)   

                                                 url  \
0  https://open.spotify.com/playlist/2JohRIjpua2F...   
1  https://open.spotify.com/playlist/2XXie4IVyrwQ...   
2  https://open.spotify.com/playlist/0PaSxJgDBs2u...   
3  https://open.spotify.com/playlist/0fHECQdMuE5d...   
4  https://open.spotify.com/playlist/7HADjFUYUmuO...   
5  https://open.spotify.com/playlist/60fG165rIvVp...   

                                         description  saves  
0  Best USA Rap out there! Featuring - Drake, Jui...  33278  
1  Songs about fireworks, explosions, sparks, USA...     79  
2                           

In [11]:
df.head()

,title,url,description,saves
0,USA Rap - 2025 🔥,https://open.spotify.com/playlist/2JohRIjpua2F...,"Best USA Rap out there! Featuring - Drake, Jui...",33278
1,"4th of July Hip Hop and R&B Indepence Day, Fi...",https://open.spotify.com/playlist/2XXie4IVyrwQ...,"Songs about fireworks, explosions, sparks, USA...",79
2,hip hop USA 2021/25,https://open.spotify.com/playlist/0PaSxJgDBs2u...,,2378
3,Dance Mix USA Vol 1,https://open.spotify.com/playlist/0fHECQdMuE5d...,A playlist collecting the original tracks of t...,1109
4,english trap/rap,https://open.spotify.com/playlist/7HADjFUYUmuO...,english trap - english rap - rap hits american...,8871


In [12]:
df.tail()

,title,url,description,saves
1,"4th of July Hip Hop and R&B Indepence Day, Fi...",https://open.spotify.com/playlist/2XXie4IVyrwQ...,"Songs about fireworks, explosions, sparks, USA...",79
2,hip hop USA 2021/25,https://open.spotify.com/playlist/0PaSxJgDBs2u...,,2378
3,Dance Mix USA Vol 1,https://open.spotify.com/playlist/0fHECQdMuE5d...,A playlist collecting the original tracks of t...,1109
4,english trap/rap,https://open.spotify.com/playlist/7HADjFUYUmuO...,english trap - english rap - rap hits american...,8871
5,USA Rap & Hip-Hop (2000-2010),https://open.spotify.com/playlist/60fG165rIvVp...,,1339


In [15]:

# Storing to CSV
df = pd.DataFrame(final_results)
df.to_csv("spotify_playlists.csv", index=False)

print("The data has been successfully saved to spotify_playlists.csv")

The data has been successfully saved to spotify_playlists.csv
